In [23]:
from pathlib import Path
BASE_DIR = Path().resolve().parent

In [24]:
file_path = [
    f"{BASE_DIR}/jars/gcs-connector.jar"
]
JARS = ",".join(file_path)

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [26]:
spark = (SparkSession.builder.appName("FlightDelay")
            .config("spark.jars", JARS)
            .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
            .config("spark.hadoop.fs.abstract_ds.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
            .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
            .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", f"{BASE_DIR}/key.json")
            .getOrCreate())

In [44]:
df.show()

+------+--------------------+
|  Code|         Description|
+------+--------------------+
|   PNM|     Panama Aviation|
|   GCH|Grand Canyon Heli...|
|   CAI|Crawford Aviation...|
|   WAI|  Wise Airlines Inc.|
|NI (1)|American Internat...|
|   DUZ|Summit Airlines Inc.|
|    A4|Southern Wind Air...|
|   NEV|     Nevada Airlines|
|   ROE|Roederer Aviation...|
|   NSQ|North Coast Air S...|
|   NKQ|           Norcanair|
|   TSQ|TPI International...|
|    LB|Lloyd Aereo Boliv...|
|   TMA|   Trans Mo Airlines|
|   BXQ|       Spantax S. A.|
|    RL|             UltrAir|
|    GJ|       Eurofly S.p.A|
|    PI|Piedmont Aviation...|
|   PBA| Provincetown Boston|
|   RDR|      Rader Aviation|
+------+--------------------+
only showing top 20 rows



In [27]:
df_airlines = spark.read.parquet("gs://flight-delay-de/bronze/airlines")
df_combined_flights = spark.read.parquet("gs://flight-delay-de/bronze/combined_flights")

In [28]:
df_airlines.show(5)

+----+--------------------+
|Code|         Description|
+----+--------------------+
| 02Q|       Titan Airways|
| 04Q|  Tradewind Aviation|
| 05Q| Comlux Aviation, AG|
| 06Q|Master Top Linhas...|
| 07Q| Flair Airlines Ltd.|
+----+--------------------+
only showing top 5 rows



In [29]:
df_airlines.filter((F.col("Code").isNull()) | (F.col("Description").isNull())).show()

+----+--------------------+
|Code|         Description|
+----+--------------------+
|NULL|North American Ai...|
+----+--------------------+



In [30]:
df_airlines.filter(F.col("Code") == 'WN').show()

+----+--------------------+
|Code|         Description|
+----+--------------------+
|  WN|Southwest Airline...|
+----+--------------------+



In [31]:
df_combined_flights.printSchema()

root
 |-- FlightDate: string (nullable = true)
 |-- Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Cancelled: string (nullable = true)
 |-- Diverted: string (nullable = true)
 |-- CRSDepTime: string (nullable = true)
 |-- DepTime: string (nullable = true)
 |-- DepDelayMinutes: string (nullable = true)
 |-- DepDelay: string (nullable = true)
 |-- ArrTime: string (nullable = true)
 |-- ArrDelayMinutes: string (nullable = true)
 |-- AirTime: string (nullable = true)
 |-- CRSElapsedTime: string (nullable = true)
 |-- ActualElapsedTime: string (nullable = true)
 |-- Distance: string (nullable = true)
 |-- Quarter: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- DayofMonth: string (nullable = true)
 |-- DayOfWeek: string (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Operated_or_Branded_Code_Share_Partners: string (nullable = true)
 |-- DOT_ID_Marketing_Airline: string (nulla

In [32]:
df_combined_flights.select("Operating_Airline").show(10)

+-----------------+
|Operating_Airline|
+-----------------+
|               WN|
|               WN|
|               WN|
|               WN|
|               WN|
|               WN|
|               WN|
|               WN|
|               WN|
|               WN|
+-----------------+
only showing top 10 rows



In [33]:
df_combined_flights.filter(F.col('Operating_Airline').isNull() | F.col("Origin").isNull() | F.col("Dest").isNull() | F.col("DepDelayMinutes").isNull()).count()

763084

In [34]:
df_combined_flights.filter(F.col("DepDelayMinutes").isNull() & (F.col("Cancelled") == True)).count()

761971

In [35]:
df_combined_flights.filter(F.col("DepDelayMinutes").isNull() & (F.col("Cancelled") == False)).count()

1113

In [36]:
df_combined_flights.filter("Cancelled == 0 AND Diverted == 0").show()

+----------+-------+------+----+---------+--------+----------+-------+---------------+--------+-------+---------------+-------+--------------+-----------------+--------+-------+-----+----------+---------+-------------------------+---------------------------------------+------------------------+---------------------------+-------------------------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+------------+---------+-------------+-------------+-------+--------+--------------------+----------+-------+---------+--------+------+----------+--------+--------+------------------+----------+-------------+------------------+----+-----------------+
|FlightDate|Airline|Origin|Dest|Cancelled|Diverted|CRSDepTime|DepTime|DepDelayMinutes|DepDelay|ArrTime|ArrDelayMinutes|AirTime|

In [37]:
df = df_combined_flights.join(df_airlines, df_combined_flights['Operating_Airline'] == df_airlines['Code'], how = 'left').drop("Code")

In [38]:
df.show(5)

+----------+--------------------+------+----+---------+--------+----------+-------+---------------+--------+-------+---------------+-------+--------------+-----------------+--------+-------+-----+----------+---------+-------------------------+---------------------------------------+------------------------+---------------------------+-------------------------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+---------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+--------------+---------+-------------+-------------+-------+--------+--------------------+----------+-------+---------+--------+------+----------+--------+--------+------------------+----------+-------------+------------------+----+-----------------+--------------------+
|FlightDate|             Airline|Origin|Dest|Cancelled|Diverted|CRSDepTime|DepTime|DepDela

In [39]:
df_combined_flights.select("OriginCityMarketID", "DestCityMarketID", "OriginWac", "DestWac", "OriginStateName", "DestStateName", "OriginState").show()

+------------------+----------------+---------+-------+---------------+-------------+-----------+
|OriginCityMarketID|DestCityMarketID|OriginWac|DestWac|OriginStateName|DestStateName|OriginState|
+------------------+----------------+---------+-------+---------------+-------------+-----------+
|             30559|           32457|       93|     91|     Washington|   California|         WA|
|             30140|           30423|       86|     74|     New Mexico|        Texas|         NM|
|             30559|           33192|       93|     91|     Washington|   California|         WA|
|             30140|           30852|       86|     35|     New Mexico|     Maryland|         NM|
|             30559|           33192|       93|     91|     Washington|   California|         WA|
|             30140|           30194|       86|     74|     New Mexico|        Texas|         NM|
|             30559|           33192|       93|     91|     Washington|   California|         WA|
|             30140|

In [47]:
df_combined_flights.groupBy('year').filter("Cancelled == false AND DepDelayMinutes IS NOT NULL").count()

28415402

In [48]:
df_combined_flights.show()

+----------+--------------------+------+----+---------+--------+----------+-------+---------------+--------+-------+---------------+-------+--------------+-----------------+--------+-------+-----+----------+---------+-------------------------+---------------------------------------+------------------------+---------------------------+-------------------------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+-----------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+--------------+---------+-------------+-------------+-------+--------+--------------------+----------+-------+---------+--------+------+----------+--------+--------+------------------+----------+-------------+------------------+----+-----------------+
|FlightDate|             Airline|Origin|Dest|Cancelled|Diverted|CRSDepTime|DepTime|DepDelayMinutes|DepDelay|A

In [40]:
df_combined_flights.select("Operated_or_Branded_Code_Share_Partners").distinct().show()

+---------------------------------------+
|Operated_or_Branded_Code_Share_Partners|
+---------------------------------------+
|                           AA_CODESHARE|
|                                     DL|
|                           AS_CODESHARE|
|                           UA_CODESHARE|
|                           DL_CODESHARE|
|                                     WN|
|                                     AA|
|                                     UA|
|                                     B6|
|                                     AS|
|                                     NK|
|                                     G4|
|                                     F9|
|                                     HA|
|                                     VX|
|                           HA_CODESHARE|
+---------------------------------------+

